In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import IntervalTrainer, SimpleTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils import InContextHead
from src import models
from src.regulariser.UnbiasRegulariser import UnbiasRegulariser
from src.regulariser.MultiRegulariser import MultiRegulariser
from src.regulariser.L2Regulariser import L2Regulariser

### Task Incremental Learning

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cpu")
head.set_context(0)
model = models.get_fully_connected_mnist_model(head=head)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Train the model on the original task.

In [ ]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Train subsequent tasks with bounds.

In [ ]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.9,
    primal_learning_rate=0.5,
    paradigm="TIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0], context_id=0)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, context_id=i)
    interval_trainer.test(test_tasks[0 : i + 1], context_list=list(range(i + 1)))
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test, context_id=i)

### Domain Incremental Training

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [ ]:
l2_reg = L2Regulariser(lmbd=0.01)
ubias_reg = UnbiasRegulariser(lmbd=0.01)
regulariser = MultiRegulariser([l2_reg, ubias_reg])

trainer = SimpleTrainer(model, domain_map_fn=domain_map_fn)
trainer.train(
    train_tasks[0],
    val_tasks[0],
    epochs=3,
)
trainer.test(test_tasks[0:1])

In [ ]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.9,
    primal_learning_rate=0.33,
    dual_learning_rate=0.01,
    batch_size=400,
    paradigm="DIL",
    domain_map_fn=domain_map_fn
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:2], val_tasks[1:2], test_tasks[1:2])
):
    interval_trainer.train(train, val, batch_size=256)
    interval_trainer.test(test_tasks[0 : i + 2])
    # if i < len(train_tasks) - 2:
    #     interval_trainer.compute_rashomon_set(test, domain_map_fn=domain_map_fn)

### Class Incremental Learning

In [4]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model()
model.to("mps")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [5]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|██████████| 3/3 [00:06<00:00,  2.19s/it, val_loss=0.0131, val_acc=0.996]


Test Results: [(0.0141, 0.996)]


[(0.0141, 0.996)]

In [ ]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.8,
    primal_learning_rate=0.33,
    dual_learning_rate=0.01,
    n_certificate_samples=400,
    paradigm="CIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:])
):
    interval_trainer.train(train, val, batch_size=64, lr=0.028, epochs=5)
    interval_trainer.test(test_tasks[0 : i + 2])
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test)

### Class Incremental Learning + Regulariser

In [6]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model()
model.to('mps')

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [7]:
unbias = UnbiasRegulariser(lmbd=0.01)
l2 = L2Regulariser(lmbd=0.01)
regulariser = MultiRegulariser([l2, unbias])

trainer = SimpleTrainer(model)
trainer.train(
    train_tasks[0], val_tasks[0], epochs=3, batch_size=256, regulariser=regulariser
)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|██████████| 3/3 [00:09<00:00,  3.17s/it, val_loss=2.03, val_acc=0.995]


Test Results: [(0.0165, 0.9945)]


[(0.0165, 0.9945)]

In [ ]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.8,
    primal_learning_rate=0.33,
    dual_learning_rate=0.01,
    n_certificate_samples=400,
    paradigm="CIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:])
):
    interval_trainer.train(train, val, batch_size=64, lr=0.028, epochs=5)
    interval_trainer.test(test_tasks[0 : i + 2])
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test)